In [1]:
"""
Real-Time Face Shape Classification using YOLO
Test Accuracy: 85% Model - Live Camera
"""

from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import torch
import os
import time


In [2]:

# ============================================
# 1. LOAD YOUR TRAINED MODEL
# ============================================
MODEL_PATH = "runs/classify/train/weights/best.pt"  

print("Loading your 85% accuracy model...")
model = YOLO("best.pt")
print("Model loaded successfully!")

CLASS_NAMES = ["Heart", "Oblong", "Oval", "Round", "Square"]

COLORS = {
    "Heart": (0, 0, 255),     
    "Oblong": (255, 0, 0),    
    "Oval": (0, 255, 0),      
    "Round": (255, 255, 0),  
    "Square": (255, 0, 255) 
}

Loading your 85% accuracy model...
Model loaded successfully!


In [3]:

# ============================================
# 2. FACE DETECTION FUNCTION (Face Detector)
# ============================================

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
def detect_face(frame):
    """كشف الوجه في الصورة"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(100, 100))
    
    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
        return (x, y, w, h)
    return None

def preprocess_face(frame, face_rect):
    """تجهيز الوجه للتصنيف"""
    x, y, w, h = face_rect
    face = frame[y:y+h, x:x+w]
    
    face_resized = cv2.resize(face, (224, 224))
    
    face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
    
    return face_rgb


In [4]:

# ============================================
# 3. CLASSIFY FACE SHAPE
# ============================================
def classify_face_shape(face_image):
    """تصنيف شكل الوجه باستخدام YOLO"""
    results = model(face_image)
    probs = results[0].probs
    
    predicted_class_idx = probs.top1
    confidence = probs.top1conf.item()
    predicted_class = CLASS_NAMES[predicted_class_idx]
    
    all_probs = probs.data.cpu().numpy()
    
    return predicted_class, confidence, all_probs


In [ ]:
"""
Face Shape Classification + Glasses Overlay
- Face detection & shape classification
- Probability bars with colors
- Auto-resize glasses overlay
"""


# ============================================
# 1. CONFIGURATION
# ============================================
MODEL_PATH = r"best.pt"  
GLASSES_BASE_DIR = r"C:\Users\hanee\Desktop\final project\glassess"

CLASS_NAMES = ["Heart", "Oblong", "Oval", "Round", "Square"]


GLASSES_SIZE_RATIO = 1.0      
GLASSES_VERTICAL_POS = -0.07 

print("="*60)
print("FACE SHAPE + GLASSES + PROBABILITY BARS")
print("="*60)

# ============================================
# 2. LOAD FACE SHAPE MODEL
# ============================================
print("\nLoading face shape model...")
try:
    model = YOLO(MODEL_PATH)
    print("Model loaded!")
except:
    print(f"Model not found at: {MODEL_PATH}")
    exit()

# ============================================
# 3. LOAD GLASSES FROM FOLDERS
# ============================================
def load_glasses_for_shape(shape_name):
    
    shape_dir = os.path.join(GLASSES_BASE_DIR, shape_name)
    glasses = []
    
    if not os.path.exists(shape_dir):
        print(f"Folder not found: {shape_dir}")
        return glasses
    
    for file in os.listdir(shape_dir):
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.PNG')):
            img_path = os.path.join(shape_dir, file)
            img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
            if img is not None:
                glasses.append({
                    'name': file,
                    'image': img
                })
                print(f"Loaded: {file}")
    
    print(f"  {shape_name}: {len(glasses)} glasses")
    return glasses

all_glasses = {}
print("\nLoading glasses...")
for shape in CLASS_NAMES:
    all_glasses[shape] = load_glasses_for_shape(shape)

# ============================================
# 4. FACE DETECTION
# ============================================
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_face(frame):
    """كشف الوجه في الصورة"""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(100, 100))
    
    if len(faces) > 0:
        return max(faces, key=lambda rect: rect[2] * rect[3])
    return None

# ============================================
# 5. CLASSIFY FACE SHAPE
# ============================================
def classify_face_shape(face_img):
    """تصنيف شكل الوجه وإرجاع الاحتمالات"""
    face_resized = cv2.resize(face_img, (160, 160))
    face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
    
    results = model(face_rgb)
    probs = results[0].probs
    pred_idx = probs.top1
    confidence = probs.top1conf.item()
    shape = CLASS_NAMES[pred_idx]
    
    
    all_probs = probs.data.cpu().numpy()
    
    return shape, confidence, all_probs

# ============================================
# 6. OVERLAY GLASSES WITH AUTO RESIZE
# ============================================
def overlay_glasses(frame, glasses_img, face_rect):
    """تركيب النظارة مع resize تلقائي حسب حجم الوجه"""
    x, y, w, h = face_rect
    
    if glasses_img is None:
        return frame
    
    
    new_width = int(w * GLASSES_SIZE_RATIO)
    
    original_h, original_w = glasses_img.shape[:2]
    aspect_ratio = original_w / original_h
    new_height = int(new_width / aspect_ratio)
    
    glasses_resized = cv2.resize(glasses_img, (new_width, new_height))
    
    glass_x = x + (w - new_width) // 2
    glass_y = y + int(h * GLASSES_VERTICAL_POS)
    
    glass_x = max(0, glass_x)
    glass_y = max(0, glass_y)
    
    frame_h, frame_w = frame.shape[:2]
    end_x = min(glass_x + new_width, frame_w)
    end_y = min(glass_y + new_height, frame_h)
    
    actual_w = end_x - glass_x
    actual_h = end_y - glass_y
    
    if actual_w <= 0 or actual_h <= 0:
        return frame
    
    glasses_resized = glasses_resized[:actual_h, :actual_w]
    
    try:
        if glasses_resized.shape[2] == 4:  
            alpha = glasses_resized[:, :, 3] / 255.0
            alpha = np.stack([alpha, alpha, alpha], axis=2)
            glass_rgb = glasses_resized[:, :, :3]
            roi = frame[glass_y:end_y, glass_x:end_x]
            
            blended = (glass_rgb * alpha + roi * (1 - alpha)).astype(np.uint8)
            frame[glass_y:end_y, glass_x:end_x] = blended
        else:
            frame[glass_y:end_y, glass_x:end_x] = glasses_resized
    except Exception as e:
        pass
    
    return frame

# =====================================
# 7. DRAW PROBABILITY BARS 
# =====================================
def draw_probability_bars(frame, all_probs, predicted_class, face_rect):
    """رسم أشرطة الاحتمالات على يمين الصورة (بنفس أسلوب الكود الأول)"""
    x, y, w, h = face_rect
    frame_h, frame_w = frame.shape[:2]
    
    bar_start_x = frame_w - 200
    bar_start_y = 60
    
    cv2.putText(frame, "Confidence Scores:", (bar_start_x, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    for i, class_name in enumerate(CLASS_NAMES):
        prob = all_probs[i] * 100
        color = COLORS.get(class_name, (128, 128, 128))
        
        bar_width = int(prob * 1.5) 
        cv2.rectangle(frame, (bar_start_x, bar_start_y + i*25),
                     (bar_start_x + bar_width, bar_start_y + i*25 + 15),
                     color, -1)
        
        
        text_color = (0, 255, 0) if class_name == predicted_class else (200, 200, 200)
        cv2.putText(frame, f"{class_name}: {prob:.1f}%",
                   (bar_start_x + 10, bar_start_y + i*25 + 12),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)

# =======================================
# 8. MAIN CAMERA FUNCTION 
# =======================================
def start_camera():
    """فتح الكاميرا مع التصنيف والنظارة وأشرطة الاحتمالات"""
    
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Cannot open camera!")
        return
    
    current_shape = None
    current_glasses_idx = 0
    current_glasses_list = []
    current_glasses_img = None
    frame_count = 0
    last_time = time.time()
    fps_counter = 0
    fps = 0
    
    print("\nCamera started!")
    print("-"*50)
    print("CONTROLS:")
    print("   'q'     - Exit")
    print("   'n'     - Next glasses")
    print("   'p'     - Previous glasses")
    print("   '+'     - Make glasses BIGGER")
    print("   '-'     - Make glasses SMALLER")
    print("-"*50)

    global GLASSES_SIZE_RATIO, GLASSES_VERTICAL_POS
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        display_frame = frame.copy()
        frame_count += 1
        
        fps_counter += 1
        current_time = time.time()
        if current_time - last_time >= 1.0:
            fps = fps_counter
            fps_counter = 0
            last_time = current_time
        
        face_rect = detect_face(frame)
        
        if face_rect is not None:
            x, y, w, h = face_rect
            
            cv2.rectangle(display_frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
            if frame_count % 30 == 0:
                face_crop = frame[y:y+h, x:x+w]
                if face_crop.size > 0:
                    shape, confidence, all_probs = classify_face_shape(face_crop)
                    
                    if shape != current_shape:
                        current_shape = shape
                        current_glasses_list = all_glasses.get(shape, [])
                        current_glasses_idx = 0
                        
                        if current_glasses_list:
                            current_glasses_img = current_glasses_list[0]['image']
                            print(f"\n👤 Face: {shape} ({confidence:.1%})")
                        else:
                            current_glasses_img = None
                            print(f"\n👤 Face: {shape} - No glasses!")
            
            if 'all_probs' in dir():
                draw_probability_bars(display_frame, all_probs, current_shape, face_rect)
            
            if current_shape:
                color = COLORS.get(current_shape, (255, 255, 255))
                
                text = f"{current_shape} ({confidence:.1%})"
                
                (text_w, text_h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
                cv2.rectangle(display_frame, (x, y-32), (x+text_w, y-5), color, -1)
                
                cv2.putText(display_frame, text, (x, y-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                confidence_bar_length = int(w * confidence)
                cv2.rectangle(display_frame, (x, y+h+5), (x+confidence_bar_length, y+h+15), color, -1)
                cv2.putText(display_frame, f"{confidence:.0%}", (x+w-40, y+h+15),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                cv2.putText(display_frame, f"Face: {w}x{h}px", 
                           (x, y+h+30), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 200, 0), 1)
            
            cv2.putText(display_frame, f"Glass size: {GLASSES_SIZE_RATIO:.1f}x | Pos: {GLASSES_VERTICAL_POS:.0%}", 
                       (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 1)
            
            if current_glasses_img is not None:
                display_frame = overlay_glasses(display_frame, current_glasses_img, face_rect)
                
                if current_glasses_list:
                    total = len(current_glasses_list)
                    cv2.putText(display_frame, f"[{current_glasses_idx+1}/{total}]", 
                               (x+w-60, y+h+30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        
        else:
            cv2.putText(display_frame, "No face detected!", (50, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        cv2.putText(display_frame, f"FPS: {fps}", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        
        
        cv2.putText(display_frame, "N:Next | P:Prev | +/-:Size | Up/Down:Pos | Q:Quit", 
                   (10, display_frame.shape[0]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        cv2.imshow('Face Shape + Glasses + Probability', display_frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('q'):
            break
        elif key == ord('n'):
            if current_glasses_list:
                current_glasses_idx = (current_glasses_idx + 1) % len(current_glasses_list)
                current_glasses_img = current_glasses_list[current_glasses_idx]['image']
                print(f"  → {current_glasses_list[current_glasses_idx]['name']}")
        elif key == ord('p'):
            if current_glasses_list:
                current_glasses_idx = (current_glasses_idx - 1) % len(current_glasses_list)
                current_glasses_img = current_glasses_list[current_glasses_idx]['image']
                print(f"  ← {current_glasses_list[current_glasses_idx]['name']}")
        elif key == ord('=') or key == ord('+'):
            GLASSES_SIZE_RATIO = min(1.8, GLASSES_SIZE_RATIO + 0.05)
            print(f"Glasses size: {GLASSES_SIZE_RATIO:.2f}x")
        elif key == ord('-') or key == ord('_'):
            GLASSES_SIZE_RATIO = max(0.7, GLASSES_SIZE_RATIO - 0.05)
            print(f"Glasses size: {GLASSES_SIZE_RATIO:.2f}x")
        
    cap.release()
    cv2.destroyAllWindows()
    print("\nCamera closed!")

# ============================================
# 9. RUN
# ============================================
if __name__ == "__main__":
    if not os.path.exists(GLASSES_BASE_DIR):
        print(f"\nError: '{GLASSES_BASE_DIR}' folder not found!")
        print(f"\nCreate folder structure:")
        for shape in CLASS_NAMES:
            print(f"   {GLASSES_BASE_DIR}/{shape}/")
        exit()
    
    start_camera()

FACE SHAPE + GLASSES + PROBABILITY BARS

Loading face shape model...
Model loaded!

Loading glasses...
Loaded: 1000121129-removebg-preview.png
Loaded: 1000121143-removebg-preview.png
Loaded: 1000121150-removebg-preview.png
Loaded: 1000121174-removebg-preview.png
  Heart: 4 glasses
Loaded: 1000121129-removebg-preview.png
Loaded: 1000121150-removebg-preview.png
Loaded: 1000121179-removebg-preview.png
  Oblong: 3 glasses
Folder not found: C:\Users\hanee\Desktop\final project\glassess\Oval
Loaded: 1000121129-removebg-preview.png
Loaded: 1000121143-removebg-preview.png
Loaded: 1000121179-removebg-preview.png
Loaded: 1000121188-removebg-preview.png
  Round: 4 glasses
Loaded: 1000121155-removebg-preview.png
Loaded: 1000121158-removebg-preview.png
Loaded: 1000121174-removebg-preview.png
Loaded: 360_F_369287574_DJpJrZCChHy090lkm50MDBhQTWLIgOuk555-removebg-preview.png
  Square: 4 glasses

Camera started!
--------------------------------------------------
CONTROLS:
   'q'     - Exit
   'n'     - 